# 🚀 ABSA Project - Full Training Pipeline

**Huấn luyện đầy đủ 6 mô hình ABSA trên Kaggle GPU (T4)**

| # | Mô hình | Kiểu | Base Model | Epochs | BS | LR |
|---|---------|------|------------|--------|----|----|
| 1 | ViSoBERT-MTL | Multi-Task | visobert | 12 | 16 | 2e-5 |
| 2 | ViSoBERT-STL | Single-Task (2 stage) | visobert | 10×2 | 16 | 2e-5 |
| 3 | PhoBERT-MTL | Multi-Task | phobert-base-v2 | 15 | 16 | 2e-5 |
| 4 | PhoBERT-STL | Single-Task (2 stage) | phobert-base-v2 | 10×2 | 16 | 2e-5 |
| 5 | BiLSTM-MTL | Multi-Task | PhoBERT embeddings | 30 | 32 | 1e-3 |
| 6 | BiLSTM-STL | Single-Task (2 stage) | PhoBERT embeddings | 25×2 | 32 | 1e-3 |

> **Unified Configuration:** seed=42, AdamW optimizer, Focal Loss (γ=2.0), max_length=256, fp16=True  
> **Tất cả model đều có Early Stopping** — training sẽ dừng sớm nếu F1 không cải thiện.

---
## 📦 Bước 1: Dọn dẹp workspace

In [ ]:
!rm -rf /kaggle/working/*

---
## 🐍 Bước 2: Cài đặt Miniconda + Python 3.10

In [ ]:
# Cài đặt Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /kaggle/working/miniconda -f
!rm /tmp/miniconda.sh

# Tạo environment Python 3.10
!/kaggle/working/miniconda/bin/conda create -y -n absa python=3.10

# Verify
!/kaggle/working/miniconda/envs/absa/bin/python --version

---
## 📥 Bước 3: Clone repository & Kiểm tra dataset

In [ ]:
# Khai báo biến activate môi trường
ACTIVATE = "source /kaggle/working/miniconda/bin/activate absa"

# Clone repo (lấy code mới nhất từ GitHub)
!git clone https://github.com/hungtran3028/ABSA-project.git /kaggle/working/ABSA-project

%cd /kaggle/working/ABSA-project

# Kiểm tra dataset
import pandas as pd
df = pd.read_csv('dataset.csv')
print(f"Dataset: {df.shape[0]} mẫu, {df.shape[1]} cột")
print(f"Cột: {list(df.columns)}")
print(f"\nPhân phối sentiment (mẫu):")
aspects = ['Battery','Camera','Performance','Display','Design','Packaging','Price','Shop_Service','Shipping','General']
for a in aspects:
    vc = df[a].value_counts()
    total = vc.sum()
    print(f"  {a:>15}: {total:>5} labeled ({vc.get('Positive',0)} pos, {vc.get('Negative',0)} neg, {vc.get('Neutral',0)} neu)")

---
## 📚 Bước 4: Cài đặt thư viện

In [ ]:
# Cài đặt PyTorch với CUDA 12.1 (tối ưu cho Kaggle T4)
!{ACTIVATE} && pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Cài đặt các thư viện cần thiết
!{ACTIVATE} && pip install transformers datasets accelerate wandb scikit-learn pandas numpy matplotlib seaborn underthesea pyvi emoji sentencepiece protobuf tqdm pyyaml statsmodels

In [ ]:
# Kiểm tra GPU & PyTorch
with open('/tmp/check_gpu.py', 'w') as f:
    f.write('''
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
''')

!{ACTIVATE} && python /tmp/check_gpu.py

---
## 📊 Bước 4.5: Cấu hình Wandb Tracking

> ⚠️ **Yêu cầu**: Thêm Kaggle Secret: `WANDB_API_KEY` (từ [wandb.ai/authorize](https://wandb.ai/authorize))

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_key
    print("✅ WANDB_API_KEY loaded from Kaggle Secrets")
except Exception as e:
    print(f"⚠️ Could not load WANDB_API_KEY: {e}")
    os.environ["WANDB_MODE"] = "offline"

!{ACTIVATE} && wandb login --relogin $WANDB_API_KEY 2>/dev/null || echo "Wandb offline mode"
os.environ["WANDB_PROJECT"] = "ABSA-Vietnamese"
print(f"📊 Wandb project: {os.environ.get('WANDB_PROJECT', 'N/A')}")

---
## 🔄 Bước 5: Chuẩn bị & Cân bằng dữ liệu

Chia dữ liệu train/val/test (80/10/10), oversampling cân bằng lớp, seed=42.

In [ ]:
!{ACTIVATE} && bash run_data_preparation.sh

In [ ]:
# Copy dữ liệu cho PhoBERT-STL và phoBERT-MTL (dùng chung split)
import shutil, os

for src, dst in {"VisoBERT-MTL/data": "phoBERT-MTL/data", "VisoBERT-STL/data": "PhoBERT-STL/data"}.items():
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✅ {src} → {dst}")
    else:
        print(f"⚠️ Not found: {src}")

---
---
# 🏋️ HUẤN LUYỆN 6 MÔ HÌNH

> ⚠️ Tất cả đều dùng Early Stopping — training sẽ dừng sớm nếu F1 không cải thiện.

## 🔵 Model 1/6: ViSoBERT-MTL (Multi-Task Learning)
- **Base**: `5CD-AI/visobert-14gb-corpus` | **Arch**: ViSoBERT → Shared Encoder → 2 Task Heads
- **Config**: Epochs 12 (early stop patience 3), BS 16, LR 2e-5, Focal Loss γ=2

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python VisoBERT-MTL/train_visobert_mtl.py --config VisoBERT-MTL/config_visobert_mtl.yaml

In [ ]:
!cat VisoBERT-MTL/models/mtl/final_report.txt

---
## 🟢 Model 2/6: ViSoBERT-STL (Single-Task Learning - 2 Stage)
- **Base**: `5CD-AI/visobert-14gb-corpus` | **Arch**: ViSoBERT → AD / SC riêng
- **Config**: Stage 1 (AD) 10 epochs + Stage 2 (SC) 10 epochs, BS 16, LR 2e-5, patience 2

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python VisoBERT-STL/train_visobert_stl.py --config VisoBERT-STL/config_visobert_stl.yaml

In [ ]:
!cat VisoBERT-STL/results/two_stage_training/final_report.txt

---
## 🔵 Model 3/6: PhoBERT-MTL (Multi-Task Learning)
- **Base**: `vinai/phobert-base-v2` | **Arch**: PhoBERT → Shared Encoder → 2 Task Heads
- **Config**: Epochs 15 (early stop patience 3), BS 16, LR 2e-5, Focal Loss γ=2

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python phoBERT-MTL/train_phobert_mtl.py --config phoBERT-MTL/config_phobert_mtl.yaml

In [ ]:
!cat phoBERT-MTL/models/mtl/final_report.txt

---
## 🟢 Model 4/6: PhoBERT-STL (Single-Task Learning - 2 Stage)
- **Base**: `vinai/phobert-base-v2` | **Arch**: PhoBERT → AD / SC riêng
- **Config**: Stage 1 (AD) 10 epochs + Stage 2 (SC) 10 epochs, BS 16, LR 2e-5, patience 5

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python PhoBERT-STL/train_phobert_stl.py --config PhoBERT-STL/config_phobert_stl.yaml

In [ ]:
!cat PhoBERT-STL/results/two_stage_training/final_report.txt

---
## 🔵 Model 5/6: BiLSTM-MTL (Multi-Task Learning)
- **Arch**: PhoBERT Embeddings (frozen) → BiLSTM → MultiHead Self-Attention → Conv1D → 2 Heads
- **Config**: Epochs 30 (early stop patience 10), BS 32, LR 1e-3, Focal Loss γ=2

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python BILSTM-MTL/train_bilstm_mtl.py --config BILSTM-MTL/config_bilstm_mtl.yaml

In [ ]:
!cat BILSTM-MTL/models/mtl/final_report.txt 2>/dev/null || echo "Report not found"

---
## 🟢 Model 6/6: BiLSTM-STL (Single-Task Learning - 2 Stage)
- **Arch**: PhoBERT Embeddings (frozen) → BiLSTM → MultiHead Self-Attention → Conv1D → Sequential
- **Config**: Stage 1 (AD) 25 epochs + Stage 2 (SC) 25 epochs, BS 32, LR 1e-3, patience 7

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python BILSTM-STL/train_two_stage_bilstm.py --config BILSTM-STL/config_bilstm_stl.yaml

In [ ]:
!cat BILSTM-STL/results/two_stage_training/final_report.txt

---
---
# 📊 PHÂN TÍCH KẾT QUẢ

> Phần này chạy tự động sau khi tất cả 6 mô hình huấn luyện xong.

## �� Bảng tổng hợp kết quả (Overall)

In [ ]:
import json, os

print("=" * 80)
print("TỔNG HỢP KẾT QUẢ HUẤN LUYỆN 6 MÔ HÌNH ABSA")
print("=" * 80)

result_paths = {
    "ViSoBERT-MTL": "VisoBERT-MTL/models/mtl/test_results.json",
    "ViSoBERT-STL": "VisoBERT-STL/models/sentiment_classification/test_results.json",
    "PhoBERT-MTL":  "phoBERT-MTL/models/mtl/test_results.json",
    "PhoBERT-STL":  "PhoBERT-STL/models/sentiment_classification/test_results.json",
    "BiLSTM-MTL":   "BILSTM-MTL/models/mtl/test_results.json",
    "BiLSTM-STL":   "BILSTM-STL/models/sentiment_classification/test_results.json",
}
ad_result_paths = {
    "ViSoBERT-STL": "VisoBERT-STL/models/aspect_detection/test_results.json",
    "PhoBERT-STL":  "PhoBERT-STL/models/aspect_detection/test_results.json",
    "BiLSTM-STL":   "BILSTM-STL/models/aspect_detection/test_results.json",
}

header = f"{'Model':<16} {'AD Acc':>8} {'AD F1':>8} {'SC Acc':>8} {'SC F1':>8}"
print(f"\n{header}")
print("-" * 52)

for model_name, sc_path in result_paths.items():
    ad_acc = ad_f1 = sc_acc = sc_f1 = "N/A"
    if os.path.exists(sc_path):
        with open(sc_path) as f:
            data = json.load(f)
        if "MTL" in model_name:
            if "ad" in data and isinstance(data["ad"], dict):
                ad_acc = f"{data['ad'].get('test_accuracy',0)*100:.2f}%"
                ad_f1 = f"{data['ad'].get('test_f1',0)*100:.2f}%"
                sc_acc = f"{data['sc'].get('test_accuracy',0)*100:.2f}%"
                sc_f1 = f"{data['sc'].get('test_f1',0)*100:.2f}%"
        else:
            sc_acc = f"{data.get('test_accuracy', data.get('accuracy',0))*100:.2f}%"
            sc_f1 = f"{data.get('test_f1', data.get('f1',0))*100:.2f}%"
    if model_name in ad_result_paths:
        ad_path = ad_result_paths[model_name]
        if os.path.exists(ad_path):
            with open(ad_path) as f:
                ad_data = json.load(f)
            ad_acc = f"{ad_data.get('test_accuracy', ad_data.get('accuracy',0))*100:.2f}%"
            ad_f1 = f"{ad_data.get('test_f1', ad_data.get('f1',0))*100:.2f}%"
    print(f"{model_name:<16} {ad_acc:>8} {ad_f1:>8} {sc_acc:>8} {sc_f1:>8}")

print("\n" + "=" * 80)

---
## 📊 Per-Aspect F1 (Chi tiết theo khía cạnh)

Phân tích F1 theo từng khía cạnh cho cả 6 mô hình — hỗ trợ §4.5 và kiểm chứng H4 (Packaging/Shipping F1 ≥ 90%).

In [ ]:
import json, os, glob

aspects = ['Battery','Camera','Performance','Display','Design','Packaging','Price','Shop_Service','Shipping','General']
models = {
    'ViSoBERT-MTL': 'VisoBERT-MTL',
    'ViSoBERT-STL': 'VisoBERT-STL',
    'PhoBERT-MTL':  'phoBERT-MTL',
    'PhoBERT-STL':  'PhoBERT-STL',
    'BiLSTM-MTL':   'BILSTM-MTL',
    'BiLSTM-STL':   'BILSTM-STL',
}

print("=" * 90)
print("PER-ASPECT F1-SCORE (AD) — Phát hiện Khía cạnh")
print("=" * 90)

for model_name, model_dir in models.items():
    # Try to find per-aspect results
    json_files = glob.glob(f"{model_dir}/**/test_results.json", recursive=True)
    for jf in json_files:
        with open(jf) as f:
            data = json.load(f)
        # Check for per-aspect data
        if "per_aspect" in data:
            print(f"\n{model_name} ({jf}):")
            for aspect, metrics in data["per_aspect"].items():
                f1 = metrics.get("f1", metrics.get("f1_score", 0)) * 100
                print(f"  {aspect:>15}: F1 = {f1:.2f}%")
        elif "ad" in data and isinstance(data["ad"], dict) and "per_aspect" in data["ad"]:
            print(f"\n{model_name} (AD):")
            for aspect, metrics in data["ad"]["per_aspect"].items():
                f1 = metrics.get("f1", 0) * 100
                print(f"  {aspect:>15}: F1 = {f1:.2f}%")
        elif "aspect_f1" in data:
            print(f"\n{model_name}:")
            for aspect, f1 in data["aspect_f1"].items():
                print(f"  {aspect:>15}: F1 = {f1*100:.2f}%")

print("\n" + "=" * 90)
print("Xem §4.5 đề cương: Packaging/Shipping cần F1 ≥ 90% (H4)")
print("=" * 90)

---
## 🔍 Error Analysis (Phân tích lỗi chi tiết)

Tạo training curves, confusion matrices, top errors, biểu đồ so sánh F1.

In [ ]:
!{ACTIVATE} && python scripts/run_error_analysis_all.py

---
## 🧮 So sánh số lượng tham số (Model Parameters)

In [ ]:
with open('/tmp/count_params.py', 'w') as f:
    f.write('''
import torch, sys, os, yaml
sys.path.insert(0, '.')

models_info = []

try:
    from transformers import AutoModel
    for name, model_name in [("ViSoBERT", "5CD-AI/visobert-14gb-corpus"), ("PhoBERT", "vinai/phobert-base-v2")]:
        m = AutoModel.from_pretrained(model_name)
        params = sum(p.numel() for p in m.parameters())
        trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
        models_info.append((name, params, trainable))
        del m
except Exception as e:
    print(f"Error loading transformer: {e}")

try:
    from BILSTM_MTL.model_bilstm_mtl import BiLSTM_MTL
    with open("BILSTM-MTL/config_bilstm_mtl.yaml") as f:
        cfg = yaml.safe_load(f)
    mc = cfg.get("model", {})
    m = BiLSTM_MTL(vocab_size=mc.get("vocab_size",64000), embedding_dim=mc.get("embedding_dim",768),
        hidden_size=mc.get("hidden_size",256), num_layers=mc.get("num_layers",2),
        num_aspects=11, num_sentiments=4, dropout=mc.get("dropout",0.3))
    params = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    models_info.append(("BiLSTM", params, trainable))
except Exception as e:
    print(f"Error loading BiLSTM: {e}")

print(f"{'Model':<15} {'Total Params':>15} {'Trainable':>15}")
print("-" * 47)
for name, total, train in models_info:
    print(f"{name:<15} {total:>15,} {train:>15,}")
''')

!{ACTIVATE} && python /tmp/count_params.py

---
## 📐 Kiểm định McNemar (Statistical Significance)

Kiểm định xem sự khác biệt giữa các mô hình có ý nghĩa thống kê hay chỉ là ngẫu nhiên.  
Hỗ trợ §4.2.3 và kiểm chứng H1, H2.

In [ ]:
!{ACTIVATE} && python scripts/run_mcnemar_test.py

---
## 📑 Tạo Bảng LaTeX tự động cho Luận văn (§4.2.1)

In [ ]:
!{ACTIVATE} && python scripts/generate_thesis_tables.py

---
## 💾 Lưu toàn bộ kết quả

In [ ]:
import shutil, os

output_dir = "/kaggle/working/ABSA-results"
os.makedirs(output_dir, exist_ok=True)

model_dirs = {
    "ViSoBERT-MTL": ["VisoBERT-MTL/models/mtl"],
    "ViSoBERT-STL": ["VisoBERT-STL/models/aspect_detection", "VisoBERT-STL/models/sentiment_classification", "VisoBERT-STL/results"],
    "PhoBERT-MTL":  ["phoBERT-MTL/models/mtl"],
    "PhoBERT-STL":  ["PhoBERT-STL/models/aspect_detection", "PhoBERT-STL/models/sentiment_classification", "PhoBERT-STL/results"],
    "BiLSTM-MTL":   ["BILSTM-MTL/models/mtl"],
    "BiLSTM-STL":   ["BILSTM-STL/models/aspect_detection", "BILSTM-STL/models/sentiment_classification", "BILSTM-STL/results"],
}

for model_name, dirs in model_dirs.items():
    for src_dir in dirs:
        if os.path.exists(src_dir):
            dst_dir = os.path.join(output_dir, src_dir)
            os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
            if os.path.exists(dst_dir): shutil.rmtree(dst_dir)
            shutil.copytree(src_dir, dst_dir)
            print(f"✅ {src_dir}")
        else:
            print(f"⚠️ Not found: {src_dir}")

if os.path.exists("error_analysis_results"):
    dst = os.path.join(output_dir, "error_analysis_results")
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree("error_analysis_results", dst)
    print("✅ error_analysis_results/")

print(f"\n📦 Tất cả kết quả đã được lưu tại: {output_dir}")

---
## ✅ Hoàn tất!

**Tất cả 6 mô hình đã được huấn luyện và phân tích.**

📁 Kết quả lưu tại `/kaggle/working/ABSA-results/`:
- `best_model.pt` — Model weights tốt nhất
- `test_results.json` — Kết quả đánh giá trên test set
- `training_history.csv` — Lịch sử training (loss, F1 theo epoch)
- `confusion_matrix_*.png` — Ma trận nhầm lẫn
- `final_report.txt` — Báo cáo chi tiết
- `error_analysis_results/` — Phân tích lỗi + biểu đồ so sánh

📊 **Wandb Dashboard**: [wandb.ai](https://wandb.ai) → project **ABSA-Vietnamese**